In [2]:
import base64
from pathlib import Path
from langchain.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing import Any, Literal
from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import RetryOutputParser
from langchain_core.exceptions import OutputParserException
from google import genai
from google.genai import types
import time

In [3]:
load_dotenv()

client = genai.Client()
video_path = "videos/B435 Intermediate Part  - 2026-08-28.mp4"
video_file = client.files.upload(file=video_path)


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
# llm_structured=llm.with_structured_output(outputSchema)

In [40]:
class ObservationItem(BaseModel):
    """A single observation item with timestamps and evidence. Each observation is a summary of of incidents within 1-3 minutes time span."""
    timestamp_start: str | None = Field(description="Start timestamp in HH:MM:SS format (e.g., '00:01:45' for 1 min 45 sec). Do not include frames.", default=None)
    timestamp_end: str | None = Field(description="End timestamp in HH:MM:SS format (e.g., '00:01:45' for 1 min 45 sec). Do not include frames.", default=None)
    evidence: str = Field(description="Evidence describing what was observed or heard. While creating the evidences make sure the names of the individuals are not mentioned, use generic terms instead.", default=None)

class Observation_agent_schema(BaseModel):
    observations: list[ObservationItem] = Field(description="List of concrete observations.")


class DimensionScores_agent_schema(BaseModel):
    """Scores for each of the four dimensions, each an integer from 1 to 4."""
    teacher_student_interaction: int = Field(ge=1, le=4, description="How effectively the teacher engaged in conversations and discussions withstudents, and how well they responded")
    verbal_instructional_clarity: int = Field(ge=1, le=4, description="How clearly the teacher explained a concept ?")
    visual_demonstration: int = Field(ge=1, le=4, description="How effectively the teacher used visual aids or demonstrations to support instruction?")
    visual_presentation: int = Field(ge=1, le=4, description="How well the lecturer present themself visually?, Was the camera of the lecturer on? Was the lecturer visible to the students? Did the lecturer have a good posture? Did the lecturer smile?")

class ObservationScoringOutput_agent_schema(BaseModel):
    dimension_scores: DimensionScores_agent_schema
    rationale: dict[Literal[
    "teacher_student_interaction",
    "verbal_instructional_clarity",
    "visual_demonstration",
    "visual_presentation",
], str]=Field(description="A rationale for each dimension score, explaining the reasoning behind the assigned score based on the observations provided.")

class Scoring_agent_schema(BaseModel):
    overall_score: float = Field(ge=1.0, le=4.0)
    overall_classification: Literal["Poor", "Average", "Good", "Excellent"]

class summarising_agent_schema(BaseModel):
    summary: str

class outputSchema(BaseModel):
    overall_classification: Literal["Poor", "Average", "Good", "Excellent"] = Field(description="Overall classification of the teaching session based on the five dimensions as Poor,Average,Good,Excellent.")
    overall_score: float = Field(description="Arithmetic mean of the five dimension scores, rounded to two decimal places.")
    summary: str = Field(description="A concise evidence-based explanation of the classification.")
    dimension_scores: dict[
      Literal[
        "teacher_student_interaction",
        "verbal_instructional_clarity",
        "visual_demonstration",
        "visual_presentation",
        "responsiveness_inclusion",
      ],
      int
    ]=Field(description="Scores for each of the five dimensions, each an integer from 1 to 4.")
    observations: str=Field(description="A list of observations, each containing timestamps and evidence. Use an empty array when there are no supported items; do not invent examples to fill an array. Use null for both observation timestamps when the time cannot be determined.")

In [5]:
# RUBRIC = """
# Evaluate only observable or audible teaching practices.

# Dimensions, scored from 1 to 4:
# 1. teacher_student_interaction: How effectively the teacher engaged in conversations and discussions withstudents, and how well they responded
# 2. verbal_instructional_clarity: How clearly the teacher explained a concept ?
# 3. visual_demonstration: How effectively the teacher used visual aids or demonstrations to support instruction?
# 4. visual_presentation: How well the lecturer present themself visually?, Was the camera of the lecturer on? Was the lecturer visible to the students? Did the lecturer have a good posture? Did the lecturer smile?


# Anchors:
# 1 = Poor: significant or repeated problems.
# 2 = Average: some effective practice with recurring gaps.
# 3 = Good: generally effective with minor gaps.
# 4 = Excellent: consistently effective with multiple clear examples.

# Use only direct evidence. Do not identify children or infer emotions, intentions,
# ability, motivation, identity, or personal characteristics. Do not invent events,
# quotes, timestamps, or context.
# """

In [6]:

observation_agent_prompt = """
<evidences to note>
1. teacher_student_interaction: How effectively the teacher engaged in conversations and discussions withstudents, and how well they responded
2. verbal_instructional_clarity: How clearly the teacher explained a concept ?
3. visual_demonstration: How effectively the teacher used visual aids or demonstrations to support instruction?
4. visual_presentation: How well the lecturer present themself visually?, Was the camera of the lecturer on? Was the lecturer visible to the students? Did the lecturer have a good posture? Did the lecturer smile?
</evidences to note>

Observations has to be made based on the evidences to note.
"""

observation_agent_prompt_template = f"""
You are the Observation Agent.

{observation_agent_prompt}

Review the supplied class recording and produce concrete observations only.

Each observation must contain:
- timestamp_start in HH:MM:SS format (e.g., '00:01:45' for 1 min 45 sec) when determinable.
- timestamp_end in HH:MM:SS format (e.g., '00:01:45' for 1 min 45 sec) when determinable.
- evidence describing what is directly observable or audible. Evidence should not include names of the individuals. Use generic terms instead.
- Consolidate related conversational exchanges into unified events (e.g. 1-3 minute segments) rather than generating an entry for every individual sentence.
- Focus specifically on teaching practices, explanations, demonstrations, and student-teacher discussions as outlined in the rubric.

Use null timestamps when time cannot be determined. Do not score the class.
If evidence is unavailable, return an empty observations list and explain the
limitation.
"""

In [7]:

try:
    max_wait = 180
    waited = 0
    while video_file.state.name == "PROCESSING":
        if waited >= max_wait:
            raise TimeoutError("Video processing timed out on Gemini server.")
        time.sleep(3)
        waited += 3
        video_file = client.files.get(name=video_file.name)

    if video_file.state.name == "FAILED":
        raise ValueError(f"Video processing failed: {video_file.error}")

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[video_file, "Review this class recording."],
        config=types.GenerateContentConfig(
            system_instruction=observation_agent_prompt_template,
            response_mime_type="application/json",
            response_schema=Observation_agent_schema,
            temperature=0,
        ),
    )
    result: Observation_agent_schema= response.parsed
    print(result)
except Exception as e:
    print(f"Error during video evaluation: {e}")
finally:
    client.files.delete(name=video_file.name)


observations=[ObservationItem(timestamp_start='00:01:41', timestamp_end='00:02:00', evidence='A student greets the teacher, and the teacher responds, stating they will wait for other students to join the class.'), ObservationItem(timestamp_start='00:02:06', timestamp_end='00:02:11', evidence='A student is visible on camera, wearing glasses, and appears to be looking down at their device.'), ObservationItem(timestamp_start='00:03:08', timestamp_end='00:03:35', evidence="The teacher appears on camera, wearing glasses, and begins the session by discussing the previous class's topic before introducing the new topic of 'King Opposition' in chess."), ObservationItem(timestamp_start='00:03:39', timestamp_end='00:04:20', evidence="The teacher explains the concept of 'King Opposition' using a chess board on screen, demonstrating how kings cannot move near each other. The teacher asks a student if they can move their king to a specific square, and the student responds."), ObservationItem(timesta

In [8]:

observations_formatted = "\n".join([
    f"[{obs.timestamp_start} - {obs.timestamp_end}]: {obs.evidence}"
    for obs in result.observations
])


SCORING_prompt_template = """
Evaluate the class on these four dimensions using only the provided observations:

Dimensions (Score 1 to 4):
1. teacher_student_interaction: How effectively the teacher engaged in conversations and discussions with students, and how well they responded.
2. verbal_instructional_clarity: How clearly the teacher explained a concept.
3. visual_demonstration: How effectively the teacher used visual aids or demonstrations to support instruction.
4. visual_presentation: How well the lecturer presented themselves visually (camera on, visible, posture, facial expression).

Anchors:
1 = Poor: Significant or repeated problems.
2 = Average: Some effective practice with recurring gaps.
3 = Good: Generally effective with minor gaps.
4 = Excellent: Consistently effective with multiple clear examples.

Provide a score (1-4) and a concrete, evidence-based rationale citing the observations for each dimension.
"""

scoring_prompt = ChatPromptTemplate.from_messages([
    ("system", f"You are an expert teaching evaluation agent.\n\n{SCORING_prompt_template}"),
    ("human", "Here are the observations recorded from the class:\n\n{observations_text}\n\nEvaluate and score each dimension.")
])

In [9]:
observation_scoring_agent = llm.with_structured_output(ObservationScoringOutput_agent_schema)
scoring_chain = scoring_prompt | observation_scoring_agent

scoring_result: ObservationScoringOutput_agent_schema = scoring_chain.invoke({"observations_text": observations_formatted})

In [18]:
print(scoring_result.rationale)

{'teacher_student_interaction': "The teacher consistently engaged students by asking questions, giving them time to think, and responding to their answers. Examples include asking a student if they could move their king (00:03:39 - 00:04:20), asking about doubts (00:06:29 - 00:06:37), prompting students to think about a position (00:12:23 - 00:12:35), and clarifying a student's suggested move (00:21:24 - 00:22:28). The teacher also welcomed new students and ensured they were caught up (00:09:05 - 00:09:22, 00:15:50 - 00:16:15).", 'verbal_instructional_clarity': "The teacher provided clear and concise explanations of complex chess concepts like 'King Opposition'. They introduced the topic effectively (00:03:08 - 00:03:35), explained why certain moves were not possible (00:04:20 - 00:06:29), and confirmed and elaborated on student answers (00:14:35 - 00:15:32). The teacher also reiterated concepts for late joiners (00:18:12 - 00:18:55) and clearly demonstrated winning/drawing strategies 

In [25]:
dimension_scores = scoring_result.dimension_scores.model_dump()
calculated_overall_score = round(sum(dimension_scores.values()) / len(dimension_scores), 2)

if calculated_overall_score >= 3.50:
    classification = "Excellent"
elif calculated_overall_score >= 2.75:
    classification = "Good"
elif calculated_overall_score >= 1.75:
    classification = "Average"
else:
    classification = "Poor"

In [26]:
print(classification)

Excellent


In [29]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an educational evaluator. Write a concise, evidence-based executive summary explaining the reason for providing the final rating."),
    ("human", """
Classification: {classification} 
calculated_overall_score: {calculated_overall_score}
Dimension Scores: {dimension_scores}
Rationales: {rationales}

Key Observations:
{observations_formatted}

Provide a concise summary paragraph.
""")
])

In [30]:
summarising_agent = llm.with_structured_output(summarising_agent_schema)

summarising_chain = summary_prompt | summarising_agent
summary_result = summarising_chain.invoke({
    "classification": classification,
    "calculated_overall_score": calculated_overall_score,
    "dimension_scores": str(dimension_scores),
    "rationales": str(scoring_result.rationale),
    "observations_formatted": observations_formatted
})

In [32]:
summary_result.summary

"The lesson received an 'Excellent' classification with an overall score of 3.75, primarily due to outstanding performance in teacher-student interaction, verbal instructional clarity, and visual demonstration. The teacher consistently engaged students through questions and thoughtful responses (e.g., 00:03:39 - 00:04:20, 00:12:23 - 00:12:35) and effectively integrated new joiners (00:09:05 - 00:09:22). Complex chess concepts like 'King Opposition' were explained with exceptional clarity and conciseness (00:03:08 - 00:03:35, 00:14:35 - 00:15:32). Visual aids, including live board demonstrations and a detailed instructional video, were expertly utilized to reinforce learning (00:03:39 - 00:04:20, 00:07:42 - 00:08:35). While visual presentation was generally effective, with the teacher visible at the start, consistent on-camera presence and expressive cues were not extensively observed, leading to a slightly lower score in that dimension. Overall, the instruction was highly effective and

In [41]:
final_output = outputSchema(
    overall_classification=classification,
    overall_score=calculated_overall_score,
    summary=summary_result.summary,
    dimension_scores=dimension_scores,
    observations=observations_formatted
)

print(final_output.model_dump_json(indent=2))

{
  "overall_classification": "Excellent",
  "overall_score": 3.75,
  "summary": "The lesson received an 'Excellent' classification with an overall score of 3.75, primarily due to outstanding performance in teacher-student interaction, verbal instructional clarity, and visual demonstration. The teacher consistently engaged students through questions and thoughtful responses (e.g., 00:03:39 - 00:04:20, 00:12:23 - 00:12:35) and effectively integrated new joiners (00:09:05 - 00:09:22). Complex chess concepts like 'King Opposition' were explained with exceptional clarity and conciseness (00:03:08 - 00:03:35, 00:14:35 - 00:15:32). Visual aids, including live board demonstrations and a detailed instructional video, were expertly utilized to reinforce learning (00:03:39 - 00:04:20, 00:07:42 - 00:08:35). While visual presentation was generally effective, with the teacher visible at the start, consistent on-camera presence and expressive cues were not extensively observed, leading to a slightly